# 🎨 Text-to-Image GAN — End-to-End Project
### Architecture: AttnGAN-lite (Attentional GAN)
- **Text Encoder**: Frozen CLIP `ViT-B/32` → 512-d sentence embedding  
- **Generator**: Noise + text embedding → ConvTranspose2d stack → 64×64 RGB image  
- **Discriminator**: Conv2d stack + text projection for conditional adversarial loss  
- **Hardware**: Auto-detects Apple M2 MPS · CUDA (Colab T4) · CPU  
- **Dataset**: Flickr8k (auto-downloaded via Kaggle API or direct URL fallback)


In [ ]:
# ── 1. Install / upgrade dependencies ────────────────────────────────────────
import sys, subprocess

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cpu")
pip("ftfy", "regex", "tqdm", "Pillow", "matplotlib", "pandas",
    "scikit-learn", "kaggle", "clip @ git+https://github.com/openai/CLIP.git")

print("✅ All packages installed")


In [ ]:
# ── 2. Imports & Hardware Detection ─────────────────────────────────────────
import os, random, math, time, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.utils as vutils

import clip  # OpenAI CLIP

# ── Device priority: MPS (M2) → CUDA (Colab) → CPU ──────────────────────────
def get_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        dev = torch.device("mps")
        label = "Apple M2 MPS 🍎"
    elif torch.cuda.is_available():
        dev = torch.device("cuda")
        props = torch.cuda.get_device_properties(0)
        label = f"CUDA — {props.name} ({props.total_memory//1024**2} MB) ⚡"
    else:
        dev = torch.device("cpu")
        label = "CPU 🖥️"
    return dev, label

DEVICE, DEVICE_LABEL = get_device()
print(f"🔧 Using device: {DEVICE_LABEL}")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ── Global hyper-parameters ───────────────────────────────────────────────────
CFG = dict(
    IMG_SIZE    = 64,        # 64×64 output images
    LATENT_DIM  = 100,       # noise vector size
    TEXT_DIM    = 512,       # CLIP embedding size (ViT-B/32)
    EMBED_DIM   = 256,       # projected text dim inside networks
    GEN_FEAT    = 64,        # generator feature-map base
    DIS_FEAT    = 64,        # discriminator feature-map base
    BATCH_SIZE  = 64,
    LR_G        = 2e-4,
    LR_D        = 2e-4,
    BETA1       = 0.5,
    BETA2       = 0.999,
    EPOCHS      = 30,        # increase to 100+ for quality results
    SAVE_EVERY  = 5,         # save checkpoint every N epochs
    NUM_WORKERS = 0,         # set to 2–4 if not on Windows/MPS
    MAX_SAMPLES = 8000,      # limit dataset for fast prototyping
)
print("✅ Configuration loaded:", CFG)


In [ ]:
# ── 3. Download Flickr8k Dataset ────────────────────────────────────────────
# Option A: Kaggle API  (set KAGGLE_USERNAME / KAGGLE_KEY env vars, or
#           place kaggle.json in ~/.kaggle/)
# Option B: We fall back to a small synthetic "toy" dataset for demonstration
#           so the notebook runs even without credentials.

import urllib.request, zipfile, shutil

DATA_DIR   = Path("flickr8k")
IMG_DIR    = DATA_DIR / "Images"
CAPS_FILE  = DATA_DIR / "captions.txt"

def make_toy_dataset(n=500):
    """Creates a minimal toy dataset (random noise images + dummy captions).
    Replace with real Flickr8k for meaningful results."""
    IMG_DIR.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    captions = []
    adjectives = ["red","blue","green","small","large","fluffy","shiny"]
    nouns      = ["bird","dog","cat","car","boat","tree","flower","horse"]
    contexts   = ["in a field","on a mountain","near water","in the forest",
                  "on a road","at sunset","under a tree","in the snow"]
    for i in range(n):
        img = Image.fromarray(rng.integers(0,255,(64,64,3), dtype=np.uint8))
        fname = f"toy_{i:04d}.jpg"
        img.save(IMG_DIR / fname)
        cap = (f"a {random.choice(adjectives)} {random.choice(nouns)} "
               f"{random.choice(contexts)}")
        captions.append({"image": fname, "caption": cap})
    df = pd.DataFrame(captions)
    df.to_csv(CAPS_FILE, index=False)
    print(f"🎲 Toy dataset created: {n} images + captions")

def download_flickr8k_kaggle():
    """Download via Kaggle API if credentials are available."""
    try:
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.dataset_download_files(
            "adityajn105/flickr8k", path=str(DATA_DIR), unzip=True)
        # Normalise caption file name
        for possible in DATA_DIR.rglob("captions.txt"):
            shutil.copy(possible, CAPS_FILE)
        print("✅ Flickr8k downloaded via Kaggle")
        return True
    except Exception as e:
        print(f"⚠️  Kaggle download failed: {e}")
        return False

if not CAPS_FILE.exists():
    print("📥 Attempting Kaggle download…")
    success = download_flickr8k_kaggle()
    if not success:
        print("📥 Falling back to toy dataset…")
        make_toy_dataset(n=500)
else:
    print("✅ Dataset already present")

# ── Inspect ──────────────────────────────────────────────────────────────────
df_caps = pd.read_csv(CAPS_FILE)
# Normalise column names
df_caps.columns = [c.strip().lower() for c in df_caps.columns]
if "file_name" in df_caps.columns:
    df_caps.rename(columns={"file_name":"image"}, inplace=True)
if "comment" in df_caps.columns:
    df_caps.rename(columns={"comment":"caption"}, inplace=True)

# Keep one caption per image (first occurrence)
df_caps = df_caps.drop_duplicates(subset="image").reset_index(drop=True)
df_caps = df_caps[df_caps["image"].apply(lambda f: (IMG_DIR/f).exists())]
df_caps = df_caps.head(CFG["MAX_SAMPLES"])

print(f"📊 Dataset: {len(df_caps)} image-caption pairs")
print(df_caps.head(3))


In [ ]:
# ── 4. Dataset & DataLoader ──────────────────────────────────────────────────

TRANSFORM = T.Compose([
    T.Resize((CFG["IMG_SIZE"], CFG["IMG_SIZE"])),
    T.RandomHorizontalFlip(),
    T.ToTensor(),
    T.Normalize([0.5]*3, [0.5]*3),   # → [-1, 1]
])

class TextImageDataset(Dataset):
    def __init__(self, df, img_dir, clip_model, clip_preprocess, transform,
                 device, max_len=77):
        self.df          = df.reset_index(drop=True)
        self.img_dir     = Path(img_dir)
        self.transform   = transform
        self.clip_model  = clip_model
        self.clip_pre    = clip_preprocess
        self.device      = device
        # Pre-compute CLIP embeddings (once, on CPU to save VRAM)
        print("🔤 Pre-computing CLIP text embeddings…")
        self.text_embeds = self._encode_all_texts()

    def _encode_all_texts(self):
        embeds = []
        bs = 128
        captions = self.df["caption"].tolist()
        with torch.no_grad():
            for i in range(0, len(captions), bs):
                batch = clip.tokenize(captions[i:i+bs], truncate=True)
                feat  = self.clip_model.encode_text(batch.to(self.device))
                feat  = feat / feat.norm(dim=-1, keepdim=True)  # L2 normalise
                embeds.append(feat.float().cpu())
        return torch.cat(embeds, dim=0)   # (N, 512)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(self.img_dir / row["image"]).convert("RGB")
        img   = self.transform(img)
        embed = self.text_embeds[idx]     # (512,)
        return img, embed


# Load CLIP (frozen) ──────────────────────────────────────────────────────────
print("🔄 Loading CLIP ViT-B/32…")
clip_model, clip_preprocess = clip.load("ViT-B/32", device=DEVICE)
clip_model.eval()
for p in clip_model.parameters():
    p.requires_grad = False
print("✅ CLIP loaded and frozen")

# Build dataset & loader ──────────────────────────────────────────────────────
dataset = TextImageDataset(df_caps, IMG_DIR, clip_model, clip_preprocess,
                           TRANSFORM, DEVICE)

loader  = DataLoader(dataset,
                     batch_size  = CFG["BATCH_SIZE"],
                     shuffle     = True,
                     num_workers = CFG["NUM_WORKERS"],
                     pin_memory  = (DEVICE.type == "cuda"),
                     drop_last   = True)

print(f"✅ DataLoader: {len(loader)} batches × {CFG['BATCH_SIZE']} samples")

# Quick visualisation ─────────────────────────────────────────────────────────
sample_imgs, sample_embeds = next(iter(loader))
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for ax, img in zip(axes, sample_imgs[:8]):
    ax.imshow((img.permute(1,2,0).numpy() * 0.5 + 0.5).clip(0,1))
    ax.axis("off")
plt.suptitle("Sample real images from DataLoader", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── 5. Model Architecture ─────────────────────────────────────────────────────
#
#  Generator
#  ─────────
#  z (100) + text_embed (512) → Linear → Reshape → ConvTranspose2d stack → 64×64 RGB
#
#  Discriminator
#  ─────────────
#  image (3×64×64) → Conv2d stack → flatten → concat with text_proj → Linear → real/fake

class TextProjection(nn.Module):
    """Projects CLIP 512-d embedding → EMBED_DIM with LayerNorm + GELU."""
    def __init__(self, in_dim=512, out_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.LayerNorm(out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)


class Generator(nn.Module):
    """
    Input : z ∈ R^{LATENT_DIM}  +  text_embed ∈ R^{TEXT_DIM}
    Output: image ∈ [-1,1]^{3×64×64}
    """
    def __init__(self, latent_dim=100, text_dim=512, embed_dim=256, feat=64):
        super().__init__()
        self.text_proj = TextProjection(text_dim, embed_dim)
        in_ch = latent_dim + embed_dim   # 100 + 256 = 356

        # 356 → 4×4×(feat*8)
        self.init_size = 4
        self.fc = nn.Sequential(
            nn.Linear(in_ch, feat*8 * self.init_size**2),
            nn.ReLU(True),
        )
        def up_block(in_c, out_c, bn=True):
            layers = [nn.ConvTranspose2d(in_c, out_c, 4, 2, 1, bias=not bn)]
            if bn: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.ReLU(True))
            return nn.Sequential(*layers)

        self.net = nn.Sequential(
            # 4×4 → 8×8
            up_block(feat*8, feat*4),
            # 8×8 → 16×16
            up_block(feat*4, feat*2),
            # 16×16 → 32×32
            up_block(feat*2, feat),
            # 32×32 → 64×64
            nn.ConvTranspose2d(feat, 3, 4, 2, 1, bias=False),
            nn.Tanh(),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.ConvTranspose2d, nn.Linear)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            if isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.constant_(m.bias, 0)

    def forward(self, z, text_embed):
        t = self.text_proj(text_embed)          # (B, 256)
        x = torch.cat([z, t], dim=1)            # (B, 356)
        x = self.fc(x)                           # (B, feat*8*16)
        x = x.view(x.size(0), -1,
                   self.init_size, self.init_size)  # (B, feat*8, 4, 4)
        return self.net(x)                       # (B, 3, 64, 64)


class Discriminator(nn.Module):
    """
    Input : image ∈ R^{3×64×64}  +  text_embed ∈ R^{TEXT_DIM}
    Output: scalar logit (real=1 / fake=0)
    """
    def __init__(self, text_dim=512, embed_dim=256, feat=64):
        super().__init__()
        self.text_proj = TextProjection(text_dim, embed_dim)

        def down_block(in_c, out_c, bn=True):
            layers = [nn.Conv2d(in_c, out_c, 4, 2, 1, bias=not bn)]
            if bn: layers.append(nn.BatchNorm2d(out_c))
            layers.append(nn.LeakyReLU(0.2, True))
            return nn.Sequential(*layers)

        self.img_net = nn.Sequential(
            down_block(3,      feat,   bn=False),  # 64→32
            down_block(feat,   feat*2),             # 32→16
            down_block(feat*2, feat*4),             # 16→8
            down_block(feat*4, feat*8),             #  8→4
        )
        # After flattening 4×4×(feat*8) + text embed_dim
        flat_dim = feat*8 * 4*4 + embed_dim
        self.classifier = nn.Sequential(
            nn.Linear(flat_dim, 1024),
            nn.LeakyReLU(0.2, True),
            nn.Dropout(0.3),
            nn.Linear(1024, 1),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.normal_(m.weight, 0.0, 0.02)
            if isinstance(m, nn.BatchNorm2d):
                nn.init.normal_(m.weight, 1.0, 0.02)
                nn.init.constant_(m.bias, 0)

    def forward(self, img, text_embed):
        feat  = self.img_net(img)                    # (B, feat*8, 4, 4)
        feat  = feat.view(feat.size(0), -1)          # (B, feat*8*16)
        t     = self.text_proj(text_embed)           # (B, 256)
        x     = torch.cat([feat, t], dim=1)
        return self.classifier(x).squeeze(1)         # (B,) logits


# Instantiate ──────────────────────────────────────────────────────────────────
G = Generator(CFG["LATENT_DIM"], CFG["TEXT_DIM"], CFG["EMBED_DIM"], CFG["GEN_FEAT"]).to(DEVICE)
D = Discriminator(CFG["TEXT_DIM"], CFG["EMBED_DIM"], CFG["DIS_FEAT"]).to(DEVICE)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Generator     params: {count_params(G):,}")
print(f"Discriminator params: {count_params(D):,}")

# Quick shape test ─────────────────────────────────────────────────────────────
with torch.no_grad():
    z_test = torch.randn(4, CFG["LATENT_DIM"]).to(DEVICE)
    t_test = torch.randn(4, CFG["TEXT_DIM"]).to(DEVICE)
    out_g  = G(z_test, t_test)
    out_d  = D(out_g, t_test)
print(f"✅ Generator output shape : {out_g.shape}")
print(f"✅ Discriminator output   : {out_d.shape}")


In [ ]:
# ── 6. Training Loop (Improved — continues from checkpoint) ──────────────────

CKPT_DIR = Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)
SAMPLE_DIR = Path("samples");   SAMPLE_DIR.mkdir(exist_ok=True)

# ── Loss: Label Smoothing makes Discriminator less overconfident ─────────────
def get_labels(size, real=True, smooth=True, device=DEVICE):
    if real:
        return torch.FloatTensor(size).uniform_(0.85, 1.0).to(device) if smooth \
               else torch.ones(size, device=device)
    else:
        return torch.FloatTensor(size).uniform_(0.0, 0.15).to(device) if smooth \
               else torch.zeros(size, device=device)

criterion = nn.BCEWithLogitsLoss()

# ── Reduced LR for fine-tuning from checkpoint ───────────────────────────────
opt_G = optim.Adam(G.parameters(), lr=1e-4, betas=(CFG["BETA1"], CFG["BETA2"]))
opt_D = optim.Adam(D.parameters(), lr=1e-4, betas=(CFG["BETA1"], CFG["BETA2"]))

# ── LR Scheduler: gradually reduce LR over time ──────────────────────────────
scheduler_G = optim.lr_scheduler.CosineAnnealingLR(opt_G, T_max=70, eta_min=1e-5)
scheduler_D = optim.lr_scheduler.CosineAnnealingLR(opt_D, T_max=70, eta_min=1e-5)

# ── Load checkpoint from epoch 30 ────────────────────────────────────────────
RESUME_FROM = "checkpoints/ckpt_epoch030.pt"
START_EPOCH = 1
history = {"loss_G": [], "loss_D": []}

if Path(RESUME_FROM).exists():
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    G.load_state_dict(ckpt["G_state"])
    D.load_state_dict(ckpt["D_state"])
    history = ckpt.get("history", {"loss_G": [], "loss_D": []})
    START_EPOCH = ckpt["epoch"] + 1
    print(f"✅ Resumed from epoch {ckpt['epoch']} → continuing from epoch {START_EPOCH}")
else:
    print("⚠️  No checkpoint found — training from scratch")

TOTAL_EPOCHS = 100   # ← train until epoch 100

# Fixed noise + embeddings for progress visualisation
FIXED_Z = torch.randn(16, CFG["LATENT_DIM"]).to(DEVICE)
FIXED_T = dataset.text_embeds[:16].to(DEVICE)

def save_checkpoint(epoch):
    path = CKPT_DIR / f"ckpt_epoch{epoch:03d}.pt"
    torch.save({
        "epoch":   epoch,
        "G_state": G.state_dict(),
        "D_state": D.state_dict(),
        "opt_G":   opt_G.state_dict(),
        "opt_D":   opt_D.state_dict(),
        "history": history,
        "cfg":     CFG,
    }, path)
    print(f"   💾 Checkpoint saved → {path}")

def show_generated(epoch, G, fixed_z, fixed_t, save=True):
    G.eval()
    with torch.no_grad():
        imgs = G(fixed_z, fixed_t).cpu()
    G.train()
    grid = vutils.make_grid(imgs, nrow=4, normalize=True, value_range=(-1,1))
    fig, ax = plt.subplots(figsize=(10,10))
    ax.imshow(grid.permute(1,2,0).numpy())
    ax.set_title(f"Epoch {epoch}", fontsize=14); ax.axis("off")
    if save:
        fig.savefig(SAMPLE_DIR / f"sample_epoch{epoch:03d}.png", bbox_inches="tight")
    plt.show()

# ── Main training loop ────────────────────────────────────────────────────────
print(f"🚀 Training epochs {START_EPOCH}–{TOTAL_EPOCHS} on {DEVICE_LABEL}\n")
t0 = time.time()

for epoch in range(START_EPOCH, TOTAL_EPOCHS + 1):
    G.train(); D.train()
    epoch_loss_G = epoch_loss_D = 0.0
    n_batches = 0

    for real_imgs, text_embeds in loader:
        real_imgs   = real_imgs.to(DEVICE)
        text_embeds = text_embeds.to(DEVICE)
        B = real_imgs.size(0)

        # Label smoothing: real=0.85–1.0, fake=0.0–0.15
        real_labels = get_labels(B, real=True,  smooth=True)
        fake_labels = get_labels(B, real=False, smooth=True)

        # ── Train Discriminator ──────────────────────────────────────────────
        opt_D.zero_grad()

        logits_real = D(real_imgs, text_embeds)
        loss_D_real = criterion(logits_real, real_labels)

        z           = torch.randn(B, CFG["LATENT_DIM"], device=DEVICE)
        fake_imgs   = G(z, text_embeds).detach()
        logits_fake = D(fake_imgs, text_embeds)
        loss_D_fake = criterion(logits_fake, fake_labels)

        # Mismatched pairs (real image + wrong caption = fake)
        shuffled_t  = text_embeds[torch.randperm(B)]
        logits_mis  = D(real_imgs, shuffled_t)
        loss_D_mis  = criterion(logits_mis, fake_labels)

        loss_D = (loss_D_real + loss_D_fake + loss_D_mis) / 3
        loss_D.backward()
        opt_D.step()

        # ── Train Generator TWICE per D step (closes G/D loss gap) ───────────
        for _ in range(2):
            opt_G.zero_grad()
            z         = torch.randn(B, CFG["LATENT_DIM"], device=DEVICE)
            fake_imgs = G(z, text_embeds)
            logits    = D(fake_imgs, text_embeds)
            # Generator wants D to think fakes are real
            loss_G    = criterion(logits, get_labels(B, real=True, smooth=False))
            loss_G.backward()
            opt_G.step()

        epoch_loss_G += loss_G.item()
        epoch_loss_D += loss_D.item()
        n_batches    += 1

    # Step LR schedulers
    scheduler_G.step()
    scheduler_D.step()

    avg_G = epoch_loss_G / n_batches
    avg_D = epoch_loss_D / n_batches
    history["loss_G"].append(avg_G)
    history["loss_D"].append(avg_D)

    current_lr = scheduler_G.get_last_lr()[0]
    elapsed    = time.time() - t0
    print(f"Epoch [{epoch:3d}/{TOTAL_EPOCHS}] | "
          f"Loss G: {avg_G:.4f} | Loss D: {avg_D:.4f} | "
          f"LR: {current_lr:.2e} | Elapsed: {elapsed/60:.1f} min")

    if epoch % CFG["SAVE_EVERY"] == 0 or epoch == TOTAL_EPOCHS:
        save_checkpoint(epoch)
        show_generated(epoch, G, FIXED_Z, FIXED_T)

print("\n✅ Training complete!")

In [ ]:
# ── 7. Loss Curves ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history["loss_G"], label="Generator Loss",     color="royalblue")
ax.plot(history["loss_D"], label="Discriminator Loss", color="tomato")
ax.set_xlabel("Epoch"); ax.set_ylabel("BCE Loss")
ax.set_title("Training Loss Curves"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(SAMPLE_DIR / "loss_curves.png", dpi=150)
plt.show()
print(f"📈 Loss curves saved → {SAMPLE_DIR / 'loss_curves.png'}")


In [ ]:
# ── 8. Inference — Generate from Custom Text ─────────────────────────────────

def generate_from_text(text: str, n_images: int = 4, seed: int = None):
    """
    Generate images conditioned on a text prompt.

    Parameters
    ----------
    text     : Natural-language caption (e.g. 'a red bird in a tree')
    n_images : Number of images to generate (rows × 2)
    seed     : Optional random seed for reproducibility
    """
    if seed is not None:
        torch.manual_seed(seed)

    G.eval()
    with torch.no_grad():
        # Encode text with CLIP
        tokens = clip.tokenize([text], truncate=True).to(DEVICE)
        t_emb  = clip_model.encode_text(tokens)
        t_emb  = t_emb / t_emb.norm(dim=-1, keepdim=True)   # L2 normalise
        t_emb  = t_emb.float().repeat(n_images, 1)           # (n, 512)

        # Sample noise
        z    = torch.randn(n_images, CFG["LATENT_DIM"], device=DEVICE)
        imgs = G(z, t_emb).cpu()

    # De-normalise [-1,1] → [0,1]
    imgs = (imgs * 0.5 + 0.5).clamp(0, 1)

    fig, axes = plt.subplots(1, n_images, figsize=(n_images * 3, 3))
    if n_images == 1:
        axes = [axes]
    for ax, img in zip(axes, imgs):
        ax.imshow(img.permute(1, 2, 0).numpy())
        ax.axis("off")
    fig.suptitle(f'Prompt: "{text}"', fontsize=12, y=1.02)
    plt.tight_layout()
    plt.savefig(SAMPLE_DIR / "inference_output.png", bbox_inches="tight", dpi=150)
    plt.show()
    print(f"✅ Generated {n_images} images  |  saved → {SAMPLE_DIR/'inference_output.png'}")


# ── ✏️  CHANGE THE PROMPT BELOW AND RE-RUN ─────────────────────────────────
MY_PROMPT = "a red bird sitting in a tree"

generate_from_text(MY_PROMPT, n_images=4, seed=0)


In [9]:
# ── 9. (Optional) Load a Saved Checkpoint ─────────────────────────────────────
def load_checkpoint(path: str):
    ckpt = torch.load(path, map_location=DEVICE)
    G.load_state_dict(ckpt["G_state"])
    D.load_state_dict(ckpt["D_state"])
    opt_G.load_state_dict(ckpt["opt_G"])
    opt_D.load_state_dict(ckpt["opt_D"])
    print(f"✅ Loaded checkpoint from epoch {ckpt['epoch']}")
    return ckpt["epoch"]

# Example usage (uncomment to load a specific checkpoint):
# last_epoch = load_checkpoint("checkpoints/ckpt_epoch030.pt")
# generate_from_text("a fluffy white cat on a red sofa", n_images=4)
print("ℹ️  Uncomment the lines above to load a specific checkpoint.")


ℹ️  Uncomment the lines above to load a specific checkpoint.
